In [1]:
print("Практическая работа №6")
print("Классификация текстов")
print("Вариант B")

Практическая работа №6
Классификация текстов
Вариант B


In [2]:
import pandas as pd
import sklearn
import nltk

print("Все библиотеки работают!")
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)

ModuleNotFoundError: No module named 'nltk'

In [3]:
%pip install nltk

  Using cached defusedxml-0.7.1-py2.py3-none-any.whl.metadata (32 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 7.5 MB/s  0:00:00 eta 0:00:01
Using cached defusedxml-0.7.1-py2.py3-none-any.whl (25 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [nltk]3/4 [nltk]

[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import sklearn
import nltk

print("Все библиотеки работают!")
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)

Все библиотеки работают!
Pandas: 3.0.2
Scikit-learn: 1.8.0


In [5]:
import pandas as pd
import os

print("Текущая папка:", os.getcwd())

Текущая папка: /Users/ainaziktogolokova/Desktop/PR1_ML/Machine_Learning/PR6/notebooks


In [6]:
df = pd.read_csv("../data/filtered.csv")

print("Размер датасета:", df.shape)
print("\nНазвания столбцов:")
print(df.columns.tolist())

df.head()

Размер датасета: (27248, 9)

Названия столбцов:
['review_id', 'product_id', 'category', 'category_label', 'rating', 'text', 'pros', 'cons', 'date']


,review_id,product_id,category,category_label,rating,text,pros,cons,date
0,1,1,electronics,Электроника,5,"Звук не плохой, максимальная громкость высокая...",Отличные наушники за свои деньги,Минусов нет,2026-06-28
1,2,1,electronics,Электроника,5,NaN,"Ничего. Подключалась 3 дня еле подключилась, н...",NaN,2026-06-26
2,3,1,electronics,Электроника,5,Сразу после первого подключения не работал вто...,NaN,NaN,2026-06-26
3,4,1,electronics,Электроника,1,Ничего нового наушники так и лежат не работают...,NaN,NaN,2026-06-23
4,5,2,electronics,Электроника,5,"Пока всё работает. Звук чистый, басы на месте,...",NaN,NaN,2026-07-19


In [7]:
# Убираем отзывы без текста и оценки
df = df.dropna(subset=["text", "rating"]).copy()

# Оставляем только положительные и отрицательные отзывы
df = df[df["rating"] != 3].copy()

# Создаём классы тональности
df["sentiment"] = df["rating"].apply(
    lambda rating: "positive" if rating >= 4 else "negative"
)

# Смотрим количество отзывов каждого класса
print("Количество отзывов по классам:")
print(df["sentiment"].value_counts())

print("\nПримеры отзывов:")
df[["text", "rating", "sentiment"]].head()

Количество отзывов по классам:
sentiment
positive    14734
negative     3258
Name: count, dtype: int64

Примеры отзывов:


,text,rating,sentiment
0,"Звук не плохой, максимальная громкость высокая...",5,positive
2,Сразу после первого подключения не работал вто...,5,positive
3,Ничего нового наушники так и лежат не работают...,1,negative
4,"Пока всё работает. Звук чистый, басы на месте,...",5,positive
5,"Когда у тебя есть дети, не расчитывай долго по...",5,positive


In [8]:
import re
import nltk
from nltk.corpus import stopwords

# Загружаем список стоп-слов для русского языка
nltk.download("stopwords")

stop_words = set(stopwords.words("russian"))

# Не удаляем отрицания — они важны для смысла отзыва
stop_words -= {"не", "ни", "нет"}

def clean_text(text):
    # Переводим текст в нижний регистр
    text = text.lower()

    # Удаляем цифры
    text = re.sub(r"\d+", " ", text)

    # Удаляем знаки препинания
    text = re.sub(r"[^\w\s]", " ", text)

    # Разбиваем текст на отдельные слова
    tokens = text.split()

    # Убираем стоп-слова
    tokens = [word for word in tokens if word not in stop_words]

    # Собираем слова обратно в текст
    return " ".join(tokens)

# Применяем очистку к отзывам
df["clean_text"] = df["text"].apply(clean_text)

# Сравниваем отзыв до и после очистки
print("ДО ОЧИСТКИ:")
print(df["text"].iloc[0][:300])

print("\nПОСЛЕ ОЧИСТКИ:")
print(df["clean_text"].iloc[0][:300])

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/ainaziktogolokova/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


ДО ОЧИСТКИ:
Звук не плохой, максимальная громкость высокая, крепко держатся и от них не болят уши

ПОСЛЕ ОЧИСТКИ:
звук не плохой максимальная громкость высокая крепко держатся не болят уши


In [9]:
print("Количество отзывов:", len(df))

print(
    "Пустых текстов после очистки:",
    (df["clean_text"].str.strip() == "").sum()
)

print("\nПримеры очищенных отзывов:")
df[["clean_text", "sentiment"]].head()

Количество отзывов: 17992
Пустых текстов после очистки: 20

Примеры очищенных отзывов:


,clean_text,sentiment
0,звук не плохой максимальная громкость высокая ...,positive
2,сразу первого подключения не работал второй на...,positive
3,нового наушники лежат не работают ребёнка отпр...,negative
4,пока всё работает звук чистый басы месте тач ч...,positive
5,дети не расчитывай долго пользоваться новыми н...,positive


In [10]:
# Убираем отзывы, которые стали пустыми после очистки
df = df[df["clean_text"].str.strip() != ""].copy()

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Разделяем данные на обучающую и тестовую части
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"],
    df["sentiment"],
    test_size=0.2,
    random_state=42,
    stratify=df["sentiment"]
)

print("Обучающих отзывов:", len(X_train))
print("Тестовых отзывов:", len(X_test))

# Создаём TF-IDF с ограничениями словаря
vectorizer = TfidfVectorizer(
    min_df=3,
    max_df=0.9
)

# Обучаем векторизатор только на тренировочных данных
X_train_tfidf = vectorizer.fit_transform(X_train)

# Преобразуем тестовые данные без повторного обучения
X_test_tfidf = vectorizer.transform(X_test)

print("\nРазмер обучающей матрицы:", X_train_tfidf.shape)
print("Размер тестовой матрицы:", X_test_tfidf.shape)
print("Размер словаря:", len(vectorizer.vocabulary_))

Обучающих отзывов: 14377
Тестовых отзывов: 3595

Размер обучающей матрицы: (14377, 11039)
Размер тестовой матрицы: (3595, 11039)
Размер словаря: 11039


In [11]:
from sklearn.naive_bayes import MultinomialNB

# Создаём модель
model = MultinomialNB()

# Обучаем модель на отзывах
model.fit(X_train_tfidf, y_train)

# Предсказываем тональность тестовых отзывов
y_pred = model.predict(X_test_tfidf)

print("Модель обучена!")
print("Количество предсказаний:", len(y_pred))

Модель обучена!
Количество предсказаний: 3595


In [12]:
from sklearn.metrics import accuracy_score, classification_report

# Считаем точность модели
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", round(accuracy, 4))

# Смотрим остальные показатели
print("\nОтчёт о качестве модели:")
print(classification_report(y_test, y_pred))

Accuracy: 0.8634

Отчёт о качестве модели:
              precision    recall  f1-score   support

    negative       0.93      0.27      0.42       652
    positive       0.86      1.00      0.92      2943

    accuracy                           0.86      3595
   macro avg       0.89      0.63      0.67      3595
weighted avg       0.87      0.86      0.83      3595



In [13]:
# Собираем реальные ответы и предсказания модели
errors = pd.DataFrame({
    "text": X_test,
    "actual": y_test,
    "predicted": y_pred
})

# Оставляем только неправильные предсказания
errors = errors[errors["actual"] != errors["predicted"]]

print("Всего ошибок:", len(errors))

# Показываем первые 5 ошибок
errors.head(5)

Всего ошибок: 491


,text,actual,predicted
8659,красила первом пользовании не сильно вовсе пер...,negative,positive
11718,треснул нескольких использования думала остави...,negative,positive
3754,кожи близко нет стелька наконечники шнурков ма...,negative,positive
26300,покупала протеин большой упаковке протеин понр...,negative,positive
15791,начиталась положительных комментариев запах ни...,positive,negative


In [14]:
for i, row in errors.head(5).iterrows():
    print(f"\nОшибка {i}")
    print("Отзыв:", row["text"])
    print("Правильный класс:", row["actual"])
    print("Предсказание:", row["predicted"])
    print("-" * 50)


Ошибка 8659
Отзыв: красила первом пользовании не сильно вовсе перестала хотя проверяла выдаче вид бала сырая красилась немного не каждый день кончики словно сухая не красит вообще цепляется ресницы такое ощущение нет высохла не советую такие деньги хлам беговая тушь р дольше служить видимо подделка ранее брала хватило полгода это вообще
Правильный класс: negative
Предсказание: positive
--------------------------------------------------

Ошибка 11718
Отзыв: треснул нескольких использования думала оставить сухих продуктов трещина стала разрастаться каждым разом контейнер оказался недолговечным стоимости непозволительно не рекомендую
Правильный класс: negative
Предсказание: positive
--------------------------------------------------

Ошибка 3754
Отзыв: кожи близко нет стелька наконечники шнурков маломерки ношу заказал р не стол брать т пальцах жмут
Правильный класс: negative
Предсказание: positive
--------------------------------------------------

Ошибка 26300
Отзыв: покупала протеин бо

## Анализ 5 ошибок модели

1. **Ошибка 8659.** Отзыв о туши для ресниц отрицательный, потому что она плохо красит и быстро перестала работать. Модель могла ошибиться из-за большого количества разных слов и сложного текста.

2. **Ошибка 11718.** Контейнер треснул после нескольких использований. Модель могла не распознать отрицательный смысл слов о поломке и недолговечности.

3. **Ошибка 3754.** Покупателю не подошёл размер обуви, она оказалась тесной. Модель могла неправильно понять отзыв из-за короткого текста и сокращений.

4. **Ошибка 26300.** Протеин оказался горьким, и покупатель больше не хочет его покупать. Модель могла ошибиться из-за того, что в отзыве есть и положительные, и отрицательные слова.

5. **Ошибка 15791.** Покупательница жалуется на запах, который не получается выветрить. Модель определила отзыв как отрицательный, хотя правильный класс в датасете — положительный. Возможно, сама оценка не полностью совпадает с текстом отзыва.